In [1]:
!pip install -q "transformers==4.44.2" "datasets==2.21.0" soundfile librosa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompat

In [2]:
import re
import numpy as np
from datasets import load_dataset, Audio
from transformers import WhisperProcessor

MODEL_NAME    = "openai/whisper-medium"
SAMPLING_RATE = 16_000
OUTPUT_DIR    = "/kaggle/working/processed_quran_words"
MAX_LABEL_LEN = 448
MAX_AUDIO_S   = 30.0

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME, language="Arabic", task="transcribe"
)

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [3]:
raw = load_dataset("Buraaq/quran-md-words")
split = raw["train"]
print(split)
print("Sample word_ar :", split[0]["word_ar"])

Generating train split:   0%|          | 0/77429 [00:00<?, ? examples/s]

Dataset({
    features: ['surah_id', 'ayah_id', 'word_id', 'word_index', 'word_ar', 'word_en', 'word_tr', 'surah_name_ar', 'surah_name_en', 'ayah_ar', 'audio'],
    num_rows: 77429
})
Sample word_ar : بِسْمِ


In [4]:
split = split.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE, mono=True))

In [5]:
# Strip only non-spoken structural marks; KEEP every diacritic.
STRUCTURAL_RE = re.compile(
    "["
    "ۖ-ۜ"   # waqf / pause marks
    "۝"          # end of ayah
    "۞"          # rub el hizb
    "۩"          # sajdah
    "ؐ-ؗ"   # honorific / annotation signs
    "ـ"          # tatweel
    "٠-٩"   # Arabic-Indic digits
    "۰-۹"   # extended Arabic-Indic digits
    "0-9"
    "]"
)

def clean_quranic_text(text: str) -> str:
    text = STRUCTURAL_RE.sub("", text)
    return re.sub(r"\s+", " ", text).strip()

for i in (0, 100, 5000):
    w = split[i]["word_ar"]
    print(repr(w), "->", repr(clean_quranic_text(w)))

'بِسْمِ' -> 'بِسْمِ'
'اللَّهَ' -> 'اللَّهَ'
'مِنْهُ' -> 'مِنْهُ'


In [6]:
def prepare_example(example):
    audio = example["audio"]
    example["audio_array"] = audio["array"].astype(np.float32)
    example["duration_s"]  = round(len(audio["array"]) / SAMPLING_RATE, 3)
    cleaned = clean_quranic_text(example["word_ar"])
    example["labels"] = processor.tokenizer(cleaned).input_ids
    example["text"]   = cleaned
    # Single audio source -> one placeholder reciter id. Keeps the schema
    # identical to Notebook 1b so the two datasets can be concatenated.
    example["reciter"] = "buraaq_words"
    return example

KEEP = {"surah_id", "ayah_id", "word_index"}
columns_to_remove = [c for c in split.column_names if c not in KEEP]

processed = split.map(
    prepare_example,
    remove_columns=columns_to_remove,
    num_proc=4,
    writer_batch_size=128,
    desc="Decoding audio + tokenizing labels",
)

Decoding audio + tokenizing labels (num_proc=4):   0%|          | 0/77429 [00:00<?, ? examples/s]

In [7]:
before = len(processed)
processed = processed.filter(
    lambda ex: 0 < len(ex["labels"]) <= MAX_LABEL_LEN
               and len(ex["text"]) > 0
               and 0.05 < ex["duration_s"] <= MAX_AUDIO_S
)
print(f"Dropped {before - len(processed)} rows; {len(processed)} remain.")

Filter:   0%|          | 0/77429 [00:00<?, ? examples/s]

Dropped 0 rows; 77429 remain.


In [8]:
durations = np.array(processed["duration_s"])
est_gb = durations.sum() * SAMPLING_RATE * 4 / 1e9
print(f"Audio: {durations.sum() / 3600:.1f} h | est on disk ~{est_gb:.1f} GB")
processed.save_to_disk(OUTPUT_DIR)
print(f"Saved {len(processed)} rows to {OUTPUT_DIR}")

Audio: 34.5 h | est on disk ~7.9 GB


Saving the dataset (0/16 shards):   0%|          | 0/77429 [00:00<?, ? examples/s]

Saved 77429 rows to /kaggle/working/processed_quran_words
